# Gold — dim_channel_scd1 (SCD 1, poc `scd_type_1` method)

One row per `channel_id`, latest values. Built like poc `dim_customer_address`: read stg → select columns → metadata → `add_checksums` → **full**: `full_refresh` (overwrite) / **incremental**: `scd_type_1` MERGE on `primary_checksum`, update only when `record_checksum` differs.

Backfill a missed day: run with `from_date` / `to_date` (Vietnam days, `to_date` exclusive). Not in the daily job yet; the live `dim_channel` is unchanged.

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = today
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
PRIMARY_KEY = ["channel_id"]
print(f"full: every read of {SILVER}.stg_channels -> {GOLD}" if LOAD_TYPE == "full"
      else f"incremental: {SILVER}.stg_channels snapshot_date in [{FROM_DATE}, {TO_DATE}) -> {GOLD}")

## Source → columns

| Column | Source (`stg_channels`) | Note |
|---|---|---|
| `channel_id` | `channel_id` | primary key |
| `channel_title` ... `published_at` | same name | descriptive attributes |
| `created_at` | `MIN(created_at)` over every read of the channel | first time the app saw the channel |
| `modified_at` | `snapshot_ts` of the last read in the window | when these values were read |
| `etl_date`, `etl_by` | job time, `Databricks` | metadata |

`created_at` / `modified_at` are **not** hashed (passed as `exclude`), so a re-read with the same values writes nothing and `etl_date` keeps meaning "changed on".

In [ ]:
first_seen = spark.table(f"{SILVER}.stg_channels").groupBy("channel_id").agg(F.min("created_at").alias("created_at"))
src = (stg_window(f"{SILVER}.stg_channels", LOAD_TYPE, FROM_DATE, TO_DATE).drop("created_at")
       .join(first_seen, "channel_id")
       .select("channel_id", "channel_title", "channel_description", "custom_url", "country", "default_language",
               "thumbnail_url", "published_at", "created_at", F.col("snapshot_ts").alias("modified_at"))
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))

src = add_checksums(src, PRIMARY_KEY, exclude=["created_at", "modified_at"])
src = src.withColumn("channel_key", F.xxhash64("channel_id"))   # same key as dim_channel, facts compute it without lookup
print(f"source rows: {src.count()}")

In [ ]:
TABLE = f"{GOLD}.dim_channel_scd1"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        channel_key BIGINT, channel_id STRING, channel_title STRING, channel_description STRING, custom_url STRING,
        country STRING, default_language STRING, thumbnail_url STRING, published_at TIMESTAMP,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Channel dimension, SCD 1 (poc checksum method). channel_key = xxhash64(channel_id), -1 = Unknown.'
""")
src = src.select(*spark.table(TABLE).columns)

if LOAD_TYPE == "full":
    src.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)   # poc full_refresh: one atomic commit
else:
    scd_type_1(src, TABLE, PRIMARY_KEY, keep=())

spark.sql(f"""
    INSERT INTO {TABLE} (channel_key, channel_title, etl_date, etl_by)
    SELECT -1, 'Unknown', current_timestamp(), 'Databricks'
    WHERE NOT EXISTS (SELECT 1 FROM {TABLE} WHERE channel_key = -1)
""")

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT channel_id) FROM {TABLE} WHERE channel_id IS NOT NULL").first()[0]
assert dup == 0, f"{TABLE}: {dup} duplicate channel_id"
print(f"✅ {TABLE}: {spark.table(TABLE).count()} rows")